# Perpetrator PCA train-only — deterministic weighted search

Versión rápida/reproducible: busca multiplicador de class weight positivo con threshold fijo 0.50.

In [1]:

# ==============================
# PERPETRATOR — PCA TRAIN-ONLY — DETERMINISTIC WEIGHT SEARCH — FIXED CALLBACK PARAMS
# ==============================
# Objetivo: reproducibilidad razonable sin grid completo.
# Estrategia:
#   - split fijo
#   - scaler/PCA fit SOLO en train
#   - semillas fijadas
#   - sin mixed precision
#   - búsqueda reducida de arquitecturas + multiplicador explícito de class_weight positivo
#   - threshold fijo = 0.50
#   - selección por criterio fijo de screening: recall_1 alto y specificity mínima

# ==============================
# CONFIGURACIÓN
# ==============================
import os
from pathlib import Path

OUTPUT_DIR = './content/perpetrator_v3'   # mantener carpeta pedida

RANDOM_STATE = 42
FORCE_CPU_FOR_REPRODUCIBILITY = False  # si quieres máxima reproducibilidad CPU: True; si tarda mucho: False
ENABLE_STRICT_OP_DETERMINISM = False   # True puede ser MUY lento

BATCH_SIZE = 128
PCA_THRESHOLD = 0.99
THRESHOLD = 0.50
EPOCHS = 80
PATIENCE = 6
LEARNING_RATE = 1e-3

# Criterio objetivo
TARGET_RECALL_POS = 0.90
MIN_RECALL_NEG = 0.20
FALLBACK_RECALL_POS = 0.85

# Multiplicadores de peso positivo. Esto reemplaza la variabilidad aleatoria por una búsqueda explícita y reproducible.
# class_weight[1] = ratio_neg_pos * POS_WEIGHT_MULTIPLIER
POS_WEIGHT_MULTIPLIERS = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

# Arquitecturas reducidas: incluye las dos candidatas que en el grid anterior llegaban al criterio,
# más variantes cercanas para no depender de una sola.
CANDIDATES = [
    (64, 'linear',  0.3,  4, 'relu',   0.10),  # grid anterior índice ~132
    (64, 'sigmoid', 0.3, 16, 'linear', 0.00),  # grid anterior índice ~329
    (64, 'linear',  0.2,  4, 'relu',   0.10),
    (64, 'linear',  0.3,  4, 'relu',   0.05),
    (64, 'relu',    0.3, 16, 'relu',   0.00),
    (64, 'relu',    0.3, 16, 'linear', 0.05),
]

TRAINING_SEEDS = [42]  # si quieres comprobar robustez, poner [42, 123], pero tardará el doble

# Si se alcanza un candidato target suficientemente bueno, salir antes.
EARLY_EXIT_IF_STRONG_TARGET = True
EARLY_EXIT_MIN_RECALL0 = 0.30

if FORCE_CPU_FOR_REPRODUCIBILITY:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

# ==============================
# IMPORTS
# ==============================
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, Callback

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
tf.keras.mixed_precision.set_global_policy('float32')

if ENABLE_STRICT_OP_DETERMINISM:
    try:
        tf.config.experimental.enable_op_determinism()
        print('TensorFlow op determinism: ON')
    except Exception as e:
        print('No se pudo activar enable_op_determinism:', repr(e))
else:
    print('TensorFlow op determinism: OFF')

print('TF version:', tf.__version__)
print('GPUs visibles:', tf.config.list_physical_devices('GPU'))
print('Política precisión:', tf.keras.mixed_precision.global_policy())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Limpiar archivos críticos para evitar leer resultados viejos si algo falla.
for fname in [
    'predictions_with_probs.csv', 'best_report.json', 'gridsearch_results.csv',
    'metrics_final.csv', 'config.json', 'best_model.h5'
]:
    p = Path(OUTPUT_DIR) / fname
    if p.exists():
        p.unlink()

# ==============================
# UTILIDADES
# ==============================
def read_csv_fallback(primary, fallback):
    if os.path.exists(primary):
        return pd.read_csv(primary)
    if os.path.exists(fallback):
        return pd.read_csv(fallback)
    raise FileNotFoundError(f'No encuentro ni {primary!r} ni {fallback!r}')


def pca_variance_df(pca_model):
    explained = pca_model.explained_variance_ratio_
    return pd.DataFrame({
        'PC': [f'PC{i+1}' for i in range(len(explained))],
        'Explained_Variance': explained,
        'Cumulative_Variance': np.cumsum(explained),
    })


def select_n_components(df_var, threshold):
    mask = df_var['Cumulative_Variance'] >= threshold
    if not mask.any():
        return len(df_var)
    return int(mask.idxmax() + 1)


def make_dataset(X, y, batch_size, seed=None, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('float32')))
    if training:
        ds = ds.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=False)
    ds = ds.batch(batch_size)
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    return ds


def build_model(input_dim, params, seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    u1, a1, d1, u2, a2, d2 = params
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(u1, activation=a1,
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 1),
                     bias_initializer='zeros'),
        layers.Dropout(d1, seed=seed + 11),
        layers.Dense(u2, activation=a2,
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 2),
                     bias_initializer='zeros'),
        layers.Dropout(d2, seed=seed + 12),
        layers.Dense(1, activation='sigmoid',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 3),
                     bias_initializer='zeros', dtype='float32'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        ],
    )
    return model


def metric_row(y_true, y_prob, params, seed, epochs_ran, pos_weight_multiplier):
    y_pred = (y_prob >= THRESHOLD).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall0 = report.get('0', {}).get('recall', 0.0)
    recall1 = report.get('1', {}).get('recall', 0.0)
    precision1 = report.get('1', {}).get('precision', 0.0)
    f1_1 = report.get('1', {}).get('f1-score', 0.0)
    accuracy = report.get('accuracy', 0.0)
    bal_acc = (recall0 + recall1) / 2
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    if recall1 >= TARGET_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 3
    elif recall1 >= FALLBACK_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 2
    elif recall1 >= TARGET_RECALL_POS:
        tier = 1
    else:
        tier = 0

    # Dentro del objetivo, preferimos mayor specificity; después PPV/balanced accuracy.
    selection_score = (tier, recall0, precision1, bal_acc, accuracy, recall1)

    u1, a1, d1, u2, a2, d2 = params
    return {
        'seed': int(seed),
        'pos_weight_multiplier': float(pos_weight_multiplier),
        'u1': u1, 'a1': a1, 'd1': d1,
        'u2': u2, 'a2': a2, 'd2': d2,
        'threshold': float(THRESHOLD),
        'recall_0': float(recall0),
        'recall_1': float(recall1),
        'specificity': float(recall0),
        'precision_ppv': float(precision1),
        'npv': float(npv),
        'f1_positive': float(f1_1),
        'accuracy': float(accuracy),
        'balanced_accuracy': float(bal_acc),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'epochs_ran': int(epochs_ran),
        'tier': int(tier),
        'selection_score': selection_score,
    }


class BestEpochByScreening(Callback):
    """Guarda los pesos del mejor epoch según el mismo criterio de selección final.

    Importante: NO usar atributo self.params porque Keras Callback lo reserva
    y lo sobrescribe durante model.fit(). Usamos self.arch_params.
    """
    def __init__(self, X_val, y_val, params, seed, pos_weight_multiplier):
        super().__init__()
        self.X_val = X_val.astype('float32')
        self.y_val = y_val.astype(int)
        self.arch_params = params  # NO usar self.params: Keras Callback lo sobrescribe internamente
        self.seed = seed
        self.pos_weight_multiplier = pos_weight_multiplier
        self.best_row = None
        self.best_weights = None
        self.best_probs = None

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.X_val, verbose=0).reshape(-1)
        row = metric_row(
            self.y_val, y_prob, self.arch_params, self.seed,
            epochs_ran=epoch + 1,
            pos_weight_multiplier=self.pos_weight_multiplier,
        )
        if self.best_row is None or row['selection_score'] > self.best_row['selection_score']:
            self.best_row = row
            self.best_weights = self.model.get_weights()
            self.best_probs = y_prob.copy()


# ==============================
# CARGA Y PREPARACIÓN DE DATOS
# ==============================
feat_df = read_csv_fallback('./../data/lista_global_vars.csv', './lista_global_vars.csv')
target_df = read_csv_fallback('./../data/target_col.csv', './target_col.csv').fillna(0)

df_merged = feat_df.join(target_df, how='inner')
df_merged = df_merged[
    ~((df_merged['GENERO_BIN_2'] == 1) | (df_merged['ORIENTSEX.BN_3'] == 1))
].drop(columns=['GENERO_BIN_2', 'ORIENTSEX.BN_3']).reset_index(drop=True)

cols_to_drop = [
    'VÍCTIMA', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION',
    'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O',
    'P.SUM.TOTAL', 'V.SUM.TOTAL'
]

df_merged_perpetrador = df_merged.drop(columns=cols_to_drop)
df_features = df_merged_perpetrador.drop(columns=['PERPETRADOR'])
y_series = df_merged_perpetrador['PERPETRADOR'].astype(int)

print('Analytical n:', len(df_features))
print('Target counts:', Counter(y_series))

df_features.to_csv(os.path.join(OUTPUT_DIR, 'df_perpetrador_feat.csv'), index=False)
y_series.to_csv(os.path.join(OUTPUT_DIR, 'df_perpretador_target.csv'), index=False)

# ==============================
# SPLIT PRIMERO; SCALER/PCA SOLO TRAIN
# ==============================
all_idx = df_features.index.to_numpy()
train_idx, val_idx = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_series,
)

with open(os.path.join(OUTPUT_DIR, 'splits_indices.json'), 'w') as f:
    json.dump({'train': train_idx.tolist(), 'val': val_idx.tolist(), 'pca_scope': 'train_only'}, f, indent=2)

X_train_raw = df_features.loc[train_idx].copy()
X_val_raw = df_features.loc[val_idx].copy()
y_train = y_series.loc[train_idx].values.astype(int)
y_val = y_series.loc[val_idx].values.astype(int)

scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

means = pd.Series(X_train_scaled.mean(axis=0), index=X_train_raw.columns, name='mean_train_scaled')
X_train_centered = X_train_scaled - means.values
X_val_centered = X_val_scaled - means.values

pca = PCA()
X_train_pca_full = pca.fit_transform(X_train_centered)
X_val_pca_full = pca.transform(X_val_centered)

df_variance = pca_variance_df(pca)
n_components = select_n_components(df_variance, PCA_THRESHOLD)

X_train = X_train_pca_full[:, :n_components].astype('float32')
X_val = X_val_pca_full[:, :n_components].astype('float32')
pc_cols = [f'PC{i+1}' for i in range(n_components)]

print(f'PCA components selected @ {PCA_THRESHOLD:.2f}:', n_components)
print(f'Train n={len(train_idx)} positives={int(y_train.sum())}; Val n={len(val_idx)} positives={int(y_val.sum())}')

joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler_minmax.pkl'))
joblib.dump(pca, os.path.join(OUTPUT_DIR, 'modelo_pca.pkl'))
means.to_csv(os.path.join(OUTPUT_DIR, 'medias_escalado.csv'), header=True)
df_variance.to_csv(os.path.join(OUTPUT_DIR, 'df_PCA_variance_trainonly.csv'), index=False)
pd.DataFrame(X_train, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
pd.DataFrame(X_val, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_val.csv'), index=False)
pd.Series(y_train, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)
pd.Series(y_val, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_val.csv'), index=False)
pd.DataFrame(np.vstack([X_train, X_val]), columns=pc_cols).to_csv(
    os.path.join(OUTPUT_DIR, f'df_PCA_{int(PCA_THRESHOLD*100)}_trainonly_stacked.csv'),
    index=False,
)

counts = Counter(y_train)
base_pos_weight = counts[0] / counts[1]
print('Base class_weight positive ratio:', base_pos_weight)
print('Total trainings:', len(CANDIDATES) * len(POS_WEIGHT_MULTIPLIERS) * len(TRAINING_SEEDS))

# ==============================
# BÚSQUEDA REDUCIDA DETERMINISTA
# ==============================
results = []
best = None
best_report = None
best_pred = None
best_probs = None
best_params = None
best_seed = None
best_pos_weight_multiplier = None
best_model_weights = None
best_model_obj = None

stop_now = False

for cand_i, params in enumerate(CANDIDATES, start=1):
    if stop_now:
        break
    for mult in POS_WEIGHT_MULTIPLIERS:
        if stop_now:
            break
        for seed_i, train_seed in enumerate(TRAINING_SEEDS, start=1):
            candidate_seed = int(train_seed + cand_i * 1000 + int(mult * 100))
            class_weight = {0: 1.0, 1: float(base_pos_weight * mult)}
            print(f'\n[cand {cand_i:02d}/{len(CANDIDATES)} mult={mult} seed={candidate_seed}] params={params} class_weight={class_weight}')

            tf.keras.backend.clear_session()
            random.seed(candidate_seed)
            np.random.seed(candidate_seed)
            tf.keras.utils.set_random_seed(candidate_seed)

            train_ds = make_dataset(X_train, y_train, BATCH_SIZE, seed=candidate_seed, training=True)
            val_ds = make_dataset(X_val, y_val, BATCH_SIZE, seed=None, training=False)
            model = build_model(X_train.shape[1], params, candidate_seed)

            best_epoch_cb = BestEpochByScreening(X_val, y_val, params, candidate_seed, mult)
            callbacks = [
                best_epoch_cb,
                EarlyStopping(monitor='val_loss', mode='min', patience=PATIENCE, restore_best_weights=False),
            ]

            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=EPOCHS,
                class_weight=class_weight,
                callbacks=callbacks,
                verbose=0,
                shuffle=False,
            )

            # Restaurar mejor epoch según criterio screening.
            if best_epoch_cb.best_weights is not None:
                model.set_weights(best_epoch_cb.best_weights)
                y_prob = best_epoch_cb.best_probs
                row = best_epoch_cb.best_row.copy()
            else:
                y_prob = model.predict(X_val.astype('float32'), verbose=0).reshape(-1)
                row = metric_row(y_val, y_prob, params, candidate_seed, len(history.history.get('loss', [])), mult)

            row['epochs_fit_total'] = int(len(history.history.get('loss', [])))
            row['base_pos_weight'] = float(base_pos_weight)
            row['effective_pos_weight'] = float(base_pos_weight * mult)
            results.append(row)

            print({k: row[k] for k in ['recall_1', 'recall_0', 'precision_ppv', 'npv', 'balanced_accuracy', 'accuracy', 'TP', 'FP', 'TN', 'FN', 'tier', 'epochs_ran', 'epochs_fit_total', 'effective_pos_weight']})

            if best is None or row['selection_score'] > best['selection_score']:
                best = row
                best_params = params
                best_seed = candidate_seed
                best_pos_weight_multiplier = mult
                best_probs = y_prob.copy()
                best_pred = (best_probs >= THRESHOLD).astype(int)
                best_report = classification_report(y_val, best_pred, output_dict=True, zero_division=0)
                best_model_weights = model.get_weights()
                best_model_obj = model
                model.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))
                print('  -> Nuevo BEST guardado')

            if (EARLY_EXIT_IF_STRONG_TARGET and row['tier'] == 3 and row['recall_0'] >= EARLY_EXIT_MIN_RECALL0):
                print('\n🚀 Strong target alcanzado; saliendo antes para ahorrar tiempo.')
                stop_now = True
                break

# ==============================
# GUARDADO FINAL
# ==============================
if best is None:
    raise RuntimeError('No se entrenó ningún candidato; revisar configuración.')

# Asegurar que el modelo guardado sea el final seleccionado.
if best_model_obj is not None and best_model_weights is not None:
    best_model_obj.set_weights(best_model_weights)
    best_model_obj.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))

# grid results
serializable_results = []
for r in results:
    rr = dict(r)
    rr.pop('selection_score', None)
    serializable_results.append(rr)

df_results = pd.DataFrame(serializable_results)
df_results.to_csv(os.path.join(OUTPUT_DIR, 'gridsearch_results.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'best_report.json'), 'w') as f:
    json.dump(best_report, f, indent=2)

# Predicciones final
df_pred = pd.DataFrame({
    'idx_original': val_idx,
    'y_true_perp': y_val.astype(int),
    'y_pred_perp': best_pred.astype(int),
    'y_prob_perp': best_probs.astype(float),
}).sort_values('idx_original').reset_index(drop=True)
df_pred.to_csv(os.path.join(OUTPUT_DIR, 'predictions_with_probs.csv'), index=False)

metrics_final = pd.DataFrame([{k: best[k] for k in [
    'threshold', 'accuracy', 'balanced_accuracy', 'precision_ppv', 'recall_1',
    'specificity', 'npv', 'f1_positive', 'TP', 'FP', 'TN', 'FN'
]}]).rename(columns={'recall_1': 'recall_sensitivity'})
metrics_final['support'] = len(y_val)
metrics_final.to_csv(os.path.join(OUTPUT_DIR, 'metrics_final.csv'), index=False)

config = {
    'random_state': RANDOM_STATE,
    'force_cpu_for_reproducibility': FORCE_CPU_FOR_REPRODUCIBILITY,
    'enable_strict_op_determinism': ENABLE_STRICT_OP_DETERMINISM,
    'threshold': THRESHOLD,
    'pca_threshold': PCA_THRESHOLD,
    'n_components': int(n_components),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'learning_rate': LEARNING_RATE,
    'target_recall_pos': TARGET_RECALL_POS,
    'min_recall_neg': MIN_RECALL_NEG,
    'fallback_recall_pos': FALLBACK_RECALL_POS,
    'base_pos_weight': float(base_pos_weight),
    'pos_weight_multipliers': POS_WEIGHT_MULTIPLIERS,
    'training_seeds': TRAINING_SEEDS,
    'candidates': [list(c) for c in CANDIDATES],
    'selection_rule': {
        'tier_3': f'recall_1 >= {TARGET_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_2': f'recall_1 >= {FALLBACK_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_1': f'recall_1 >= {TARGET_RECALL_POS}',
        'score': '(tier, recall_0, precision_ppv, balanced_accuracy, accuracy, recall_1)',
        'threshold_fixed': THRESHOLD,
        'best_epoch_selected_by_same_screening_score': True,
    },
    'best': {k: v for k, v in best.items() if k != 'selection_score'},
    'best_params': {
        'u1': best_params[0], 'a1': best_params[1], 'd1': best_params[2],
        'u2': best_params[3], 'a2': best_params[4], 'd2': best_params[5],
        'seed': int(best_seed),
        'pos_weight_multiplier': float(best_pos_weight_multiplier),
        'effective_pos_weight': float(base_pos_weight * best_pos_weight_multiplier),
    },
}
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('\n=== BEST SELECTED ===')
print(pd.Series({k: v for k, v in best.items() if k != 'selection_score'}))
print('\n=== FINAL REPORT — PERPETRATOR CSV ===')
print(classification_report(y_val, best_pred, digits=3, zero_division=0))
print('\n=== TOP 10 GRID RESULTS ===')
print(df_results.sort_values(['tier','recall_0','precision_ppv','balanced_accuracy'], ascending=False).head(10).to_string(index=False))
print('\nGuardado en:', OUTPUT_DIR)
print('Archivos principales: predictions_with_probs.csv, best_report.json, gridsearch_results.csv, metrics_final.csv, config.json, best_model.h5')


TensorFlow op determinism: OFF
TF version: 2.21.0
GPUs visibles: []
Política precisión: <DTypePolicy "float32">
Analytical n: 3767
Target counts: Counter({0: 2882, 1: 885})
PCA components selected @ 0.99: 22
Train n=2825 positives=664; Val n=942 positives=221
Base class_weight positive ratio: 3.2545180722891565
Total trainings: 42

[cand 01/6 mult=1.0 seed=1142] params=(64, 'linear', 0.3, 4, 'relu', 0.1) class_weight={0: 1.0, 1: 3.2545180722891565}


{'recall_1': 0.6199095022624435, 'recall_0': 0.6796116504854369, 'precision_ppv': 0.37228260869565216, 'npv': 0.8536585365853658, 'balanced_accuracy': 0.6497605763739402, 'accuracy': 0.6656050955414012, 'TP': 137, 'FP': 231, 'TN': 490, 'FN': 84, 'tier': 0, 'epochs_ran': 12, 'epochs_fit_total': 22, 'effective_pos_weight': 3.2545180722891565}
  -> Nuevo BEST guardado

[cand 01/6 mult=1.5 seed=1192] params=(64, 'linear', 0.3, 4, 'relu', 0.1) class_weight={0: 1.0, 1: 4.8817771084337345}


{'recall_1': 0.9004524886877828, 'recall_0': 0.2912621359223301, 'precision_ppv': 0.2802816901408451, 'npv': 0.9051724137931034, 'balanced_accuracy': 0.5958573123050565, 'accuracy': 0.43418259023354566, 'TP': 199, 'FP': 511, 'TN': 210, 'FN': 22, 'tier': 3, 'epochs_ran': 2, 'epochs_fit_total': 7, 'effective_pos_weight': 4.8817771084337345}
  -> Nuevo BEST guardado

[cand 01/6 mult=2.0 seed=1242] params=(64, 'linear', 0.3, 4, 'relu', 0.1) class_weight={0: 1.0, 1: 6.509036144578313}
{'recall_1': 0.995475113122172, 'recall_0': 0.09153952843273232, 'precision_ppv': 0.25142857142857145, 'npv': 0.9850746268656716, 'balanced_accuracy': 0.5435073207774521, 'accuracy': 0.3036093418259023, 'TP': 220, 'FP': 655, 'TN': 66, 'FN': 1, 'tier': 1, 'epochs_ran': 7, 'epochs_fit_total': 7, 'effective_pos_weight': 6.509036144578313}

[cand 01/6 mult=2.5 seed=1292] params=(64, 'linear', 0.3, 4, 'relu', 0.1) class_weight={0: 1.0, 1: 8.136295180722891}
{'recall_1': 0.9864253393665159, 'recall_0': 0.14563106796

{'recall_1': 0.9140271493212669, 'recall_0': 0.2926490984743412, 'precision_ppv': 0.28370786516853935, 'npv': 0.9173913043478261, 'balanced_accuracy': 0.6033381238978041, 'accuracy': 0.43842887473460723, 'TP': 202, 'FP': 510, 'TN': 211, 'FN': 19, 'tier': 3, 'epochs_ran': 5, 'epochs_fit_total': 8, 'effective_pos_weight': 4.8817771084337345}
  -> Nuevo BEST guardado

[cand 04/6 mult=2.0 seed=4242] params=(64, 'linear', 0.3, 4, 'relu', 0.05) class_weight={0: 1.0, 1: 6.509036144578313}


{'recall_1': 0.918552036199095, 'recall_0': 0.29958391123439665, 'precision_ppv': 0.2867231638418079, 'npv': 0.9230769230769231, 'balanced_accuracy': 0.6090679737167458, 'accuracy': 0.4447983014861996, 'TP': 203, 'FP': 505, 'TN': 216, 'FN': 18, 'tier': 3, 'epochs_ran': 7, 'epochs_fit_total': 7, 'effective_pos_weight': 6.509036144578313}
  -> Nuevo BEST guardado

[cand 04/6 mult=2.5 seed=4292] params=(64, 'linear', 0.3, 4, 'relu', 0.05) class_weight={0: 1.0, 1: 8.136295180722891}
{'recall_1': 0.9728506787330317, 'recall_0': 0.15672676837725383, 'precision_ppv': 0.26123936816524906, 'npv': 0.9495798319327731, 'balanced_accuracy': 0.5647887235551428, 'accuracy': 0.3481953290870488, 'TP': 215, 'FP': 608, 'TN': 113, 'FN': 6, 'tier': 1, 'epochs_ran': 7, 'epochs_fit_total': 7, 'effective_pos_weight': 8.136295180722891}

[cand 04/6 mult=3.0 seed=4342] params=(64, 'linear', 0.3, 4, 'relu', 0.05) class_weight={0: 1.0, 1: 9.763554216867469}
{'recall_1': 0.9909502262443439, 'recall_0': 0.015256588

{'recall_1': 0.9140271493212669, 'recall_0': 0.30097087378640774, 'precision_ppv': 0.28611898016997167, 'npv': 0.9194915254237288, 'balanced_accuracy': 0.6074990115538373, 'accuracy': 0.4447983014861996, 'TP': 202, 'FP': 504, 'TN': 217, 'FN': 19, 'tier': 3, 'epochs_ran': 4, 'epochs_fit_total': 27, 'effective_pos_weight': 4.8817771084337345}
  -> Nuevo BEST guardado

🚀 Strong target alcanzado; saliendo antes para ahorrar tiempo.

=== BEST SELECTED ===
seed                         6192
pos_weight_multiplier         1.5
u1                             64
a1                           relu
d1                            0.3
u2                             16
a2                         linear
d2                           0.05
threshold                     0.5
recall_0                 0.300971
recall_1                 0.914027
specificity              0.300971
precision_ppv            0.286119
npv                      0.919492
f1_positive              0.435814
accuracy                 0.444798
b

In [2]:

# CHECK opcional: ejecutar SOLO después de que haya terminado la celda anterior.
import json
from pprint import pprint
import pandas as pd
from sklearn.metrics import classification_report

pred_path = './content/perpetrator_v3/predictions_with_probs.csv'
data_set = pd.read_csv(pred_path)

print('=== CHECK predictions_with_probs.csv ===')
print(data_set.head())
print(data_set.shape)
print(list(data_set.columns))
print(classification_report(data_set['y_true_perp'], data_set['y_pred_perp'], digits=3, zero_division=0))

print('\n=== METRICS FINAL ===')
print(pd.read_csv('./content/perpetrator_v3/metrics_final.csv').T)

print('\n=== BEST CONFIG ===')
with open('./content/perpetrator_v3/config.json', 'r') as f:
    config = json.load(f)
pprint(config['best_params'])
print('\nBEST METRICS:')
pprint(config['best'])

print('\n=== TOP 10 GRID RESULTS ===')
df_grid = pd.read_csv('./content/perpetrator_v3/gridsearch_results.csv')
print(df_grid.sort_values(['tier','recall_0','precision_ppv','balanced_accuracy'], ascending=False).head(10).to_string(index=False))


=== CHECK predictions_with_probs.csv ===
   idx_original  y_true_perp  y_pred_perp  y_prob_perp
0             1            1            1     0.554482
1             2            0            1     0.636366
2             5            0            1     0.659102
3            11            0            1     0.502891
4            13            0            1     0.606305
(942, 4)
['idx_original', 'y_true_perp', 'y_pred_perp', 'y_prob_perp']
              precision    recall  f1-score   support

           0      0.919     0.301     0.454       721
           1      0.286     0.914     0.436       221

    accuracy                          0.445       942
   macro avg      0.603     0.607     0.445       942
weighted avg      0.771     0.445     0.449       942


=== METRICS FINAL ===
                             0
threshold             0.500000
accuracy              0.444798
balanced_accuracy     0.607499
precision_ppv         0.286119
recall_sensitivity    0.914027
specificity           

In [3]:
# ============================================================
# PERPETRATION — MODEL COMPARISON USING FINAL PCA TRAIN-ONLY FILES
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
THRESHOLD = 0.5

# CAMBIA ESTA RUTA si tu carpeta final se llama diferente
perp_dir = Path("./content/perpetrator_v3").resolve()

comparison_dir = Path("./content/perp_model_comparison_PCA_trainonly").resolve()
comparison_dir.mkdir(parents=True, exist_ok=True)

print("CWD:", os.getcwd())
print("Perpetration dir:", perp_dir)
print("Comparison dir:", comparison_dir)

# ------------------------------------------------------------
# 1. Load final PCA train-only files
# ------------------------------------------------------------

X_train_path = perp_dir / "X_train.csv"
X_test_path  = perp_dir / "X_val.csv"
y_train_path = perp_dir / "y_train.csv"
y_test_path  = perp_dir / "y_val.csv"
pred_path    = perp_dir / "predictions_with_probs.csv"

print("\nChecking files:")
for p in [X_train_path, X_test_path, y_train_path, y_test_path, pred_path]:
    print(p.name, p.exists(), p)

def load_y_vector(path):
    """
    Loads y_train/y_val/y_test saved in different possible CSV formats:
    - one unnamed column
    - index + one column
    - headerless vector
    - column named y / target / etc.
    """
    path = Path(path)

    df = pd.read_csv(path)

    print(f"\nLoading {path.name}")
    print("Raw shape:", df.shape)
    print("Raw columns:", list(df.columns))
    print(df.head())

    # Remove common index-like columns
    index_like_cols = [
        c for c in df.columns
        if str(c).lower().startswith("unnamed")
        or str(c).lower() in ["index", "idx", "id"]
    ]

    if index_like_cols and df.shape[1] > 1:
        df = df.drop(columns=index_like_cols)

    if df.shape[1] == 0:
        df = pd.read_csv(path, header=None)
        print("Fallback header=None shape:", df.shape)

    if df.shape[1] > 1:
        possible_target_cols = [
            c for c in df.columns
            if str(c).lower() in [
                "y", "target", "perp", "perpetrador",
                "y_train", "y_val", "y_test", "y_true_perp"
            ]
        ]
        if possible_target_cols:
            s = df[possible_target_cols[0]]
        else:
            s = df.iloc[:, -1]
    else:
        s = df.iloc[:, 0]

    return s.astype(int).to_numpy()


# Load X
X_train_cmp = pd.read_csv(X_train_path, index_col=0)
X_test_cmp  = pd.read_csv(X_test_path, index_col=0)

# Load y
y_train_cmp = load_y_vector(y_train_path)
y_test_cmp  = load_y_vector(y_test_path)

# Convert X to numpy
X_train_np = X_train_cmp.to_numpy()
X_test_np = X_test_cmp.to_numpy()

print("\nShapes:")
print("X_train:", X_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_train:", y_train_cmp.shape, "positives:", y_train_cmp.sum())
print("y_test:", y_test_cmp.shape, "positives:", y_test_cmp.sum())


# ------------------------------------------------------------
# 2. Metric helper
# ------------------------------------------------------------

def compute_binary_metrics(y_true, y_pred, model_name, threshold=0.5):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    return {
        "outcome": "perpetration",
        "model": model_name,
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
        "support": int(len(y_true)),
        "positive_support": int(np.sum(y_true == 1)),
        "negative_support": int(np.sum(y_true == 0)),
    }


def fit_predict_metrics(model_name, model, X_train, y_train, X_test, y_test, threshold=0.5, sample_weight=None):
    model = clone(model)

    if sample_weight is not None:
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)
    else:
        y_pred = model.predict(X_test).astype(int)

    return compute_binary_metrics(y_test, y_pred, model_name, threshold)


# ------------------------------------------------------------
# 3. Include selected final DNN from frozen predictions
# ------------------------------------------------------------

rows = []

if pred_path.exists():
    pred_df = pd.read_csv(pred_path)

    # Expected columns:
    # idx_original, y_true_perp, y_pred_perp, y_prob_perp
    print("\nPredictions columns:", list(pred_df.columns))

    y_true_final = pred_df["y_true_perp"].astype(int).to_numpy()
    y_pred_final = pred_df["y_pred_perp"].astype(int).to_numpy()

    # Safety check
    if len(y_true_final) != len(y_test_cmp):
        print("WARNING: predictions length differs from y_test length.")
    if not np.array_equal(y_true_final, y_test_cmp):
        print("WARNING: y_true_perp in predictions does not exactly match y_val.csv order.")
        print("Using y_true_perp from predictions for DNN final row only.")

    rows.append(
        compute_binary_metrics(
            y_true=y_true_final,
            y_pred=y_pred_final,
            model_name="DNN_selected_FINAL",
            threshold=THRESHOLD
        )
    )

    print("Loaded selected final DNN predictions:", pred_path)
else:
    print("WARNING: predictions_with_probs.csv not found:", pred_path)


# ------------------------------------------------------------
# 4. Candidate classical models
# ------------------------------------------------------------

balanced_sw = compute_sample_weight(class_weight="balanced", y=y_train_cmp)

# Extra positive weights for perpetration screening.
# This helps check whether classical models can reach high sensitivity
# under a screening-oriented class weighting scheme.
pos_rate = y_train_cmp.mean()
neg_rate = 1 - pos_rate
base_pos_weight = neg_rate / pos_rate

sw_pos_1_5 = np.where(y_train_cmp == 1, base_pos_weight * 1.5, 1.0)
sw_pos_2_0 = np.where(y_train_cmp == 1, base_pos_weight * 2.0, 1.0)
sw_pos_3_0 = np.where(y_train_cmp == 1, base_pos_weight * 3.0, 1.0)

print("\nBase positive weight:", base_pos_weight)

candidate_models = [
    ("Dummy_most_frequent", DummyClassifier(strategy="most_frequent"), None),

    ("LogisticRegression", LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE), None),
    ("LogisticRegression_balanced", LogisticRegression(max_iter=5000, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE), None),
    ("LogisticRegression_SW_pos1.5", LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE), sw_pos_1_5),
    ("LogisticRegression_SW_pos2.0", LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE), sw_pos_2_0),
    ("LogisticRegression_SW_pos3.0", LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE), sw_pos_3_0),

    ("DecisionTree_unpruned", DecisionTreeClassifier(random_state=RANDOM_STATE), None),
    ("DecisionTree_balanced_unpruned", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"), None),

    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1), None),
    ("RandomForest_balanced", RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"), None),

    ("GradientBoosting", GradientBoostingClassifier(random_state=RANDOM_STATE), None),
    ("GradientBoosting_balancedSW", GradientBoostingClassifier(random_state=RANDOM_STATE), balanced_sw),
    ("GradientBoosting_SW_pos1.5", GradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_1_5),
    ("GradientBoosting_SW_pos2.0", GradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_2_0),
    ("GradientBoosting_SW_pos3.0", GradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_3_0),

    ("HistGradientBoosting", HistGradientBoostingClassifier(random_state=RANDOM_STATE), None),
    ("HistGradientBoosting_balancedSW", HistGradientBoostingClassifier(random_state=RANDOM_STATE), balanced_sw),
    ("HistGradientBoosting_SW_pos1.5", HistGradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_1_5),
    ("HistGradientBoosting_SW_pos2.0", HistGradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_2_0),
    ("HistGradientBoosting_SW_pos3.0", HistGradientBoostingClassifier(random_state=RANDOM_STATE), sw_pos_3_0),
]

for model_name, model, sw in candidate_models:
    print("Running:", model_name)
    row = fit_predict_metrics(
        model_name=model_name,
        model=model,
        X_train=X_train_np,
        y_train=y_train_cmp,
        X_test=X_test_np,
        y_test=y_test_cmp,
        threshold=THRESHOLD,
        sample_weight=sw
    )
    rows.append(row)


# ------------------------------------------------------------
# 5. Results table
# ------------------------------------------------------------

comparison_df = pd.DataFrame(rows)

comparison_df["meets_recall_085"] = comparison_df["recall_sensitivity"] >= 0.85
comparison_df["meets_recall_090"] = comparison_df["recall_sensitivity"] >= 0.90

# For perpetration we prioritize sensitivity around/above 90%.
# Among high-recall models, we prefer better specificity and balanced accuracy.
comparison_df = comparison_df.sort_values(
    by=[
        "meets_recall_090",
        "meets_recall_085",
        "balanced_accuracy",
        "specificity",
        "precision_ppv"
    ],
    ascending=[False, False, False, False, False]
).reset_index(drop=True)

comparison_path = comparison_dir / "perp_model_comparison_PCA_trainonly.csv"
comparison_df.to_csv(comparison_path, index=False)

display_cols = [
    "model", "threshold",
    "TP", "FP", "TN", "FN",
    "recall_sensitivity", "specificity", "precision_ppv", "npv",
    "balanced_accuracy", "f1_positive", "accuracy",
    "meets_recall_085", "meets_recall_090"
]

print("\n=== PERPETRATION MODEL COMPARISON — PCA TRAIN-ONLY ===")
print(
    comparison_df[display_cols]
    .to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

print("\nSaved to:", comparison_path)

CWD: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator
Perpetration dir: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3
Comparison dir: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perp_model_comparison_PCA_trainonly

Checking files:
X_train.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/X_train.csv
X_val.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/X_val.csv
y_train.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/y_train.csv
y_val.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version

In [4]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

display_cols = [
    "model", "threshold",
    "TP", "FP", "TN", "FN",
    "recall_sensitivity", "specificity", "precision_ppv", "npv",
    "balanced_accuracy", "f1_positive", "accuracy",
    "meets_recall_085", "meets_recall_090"
]

print(
    comparison_df[display_cols]
    .to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

                          model  threshold  TP  FP  TN  FN  recall_sensitivity  specificity  precision_ppv   npv  balanced_accuracy  f1_positive  accuracy  meets_recall_085  meets_recall_090
             DNN_selected_FINAL      0.500 202 504 217  19               0.914        0.301          0.286 0.919              0.607        0.436     0.445              True              True
   LogisticRegression_SW_pos3.0      0.500 212 585 136   9               0.959        0.189          0.266 0.938              0.574        0.417     0.369              True              True
   LogisticRegression_SW_pos2.0      0.500 194 472 249  27               0.878        0.345          0.291 0.902              0.612        0.437     0.470              True             False
    LogisticRegression_balanced      0.500 134 220 501  87               0.606        0.695          0.379 0.852              0.651        0.466     0.674             False             False
   LogisticRegression_SW_pos1.5      0.500 17

In [5]:
# ============================================================
# EXPORT ALIGNED TEST ROW-LEVEL DATA
# Run at the END of each final notebook
# Change only OUTCOME
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG — change this in each notebook
# ------------------------------------------------------------

#OUTCOME = "victimization"
OUTCOME = "perpetration"
# OUTCOME = "overlap"

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"

SUBGROUP_COLS = [
    "PAÍS",
    "ETNIA.BN",
    "EDAD",
    "GENERO_BIN_1",
    "GENERO_BIN_2",
    "ORIENTSEX.BN_1",
    "ORIENTSEX.BN_2",
    "ORIENTSEX.BN_3",
]

# Find Version_Final automatically
VERSION_FINAL_DIR = Path.cwd().resolve()
while VERSION_FINAL_DIR.name != "Version_Final" and VERSION_FINAL_DIR.parent != VERSION_FINAL_DIR:
    VERSION_FINAL_DIR = VERSION_FINAL_DIR.parent

if VERSION_FINAL_DIR.name != "Version_Final":
    raise RuntimeError("Could not locate Version_Final directory. Run from inside Version_Final or a subfolder.")

EXPORT_DIR = VERSION_FINAL_DIR / "aligned_test_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_FILES = {
    "victimization": VERSION_FINAL_DIR / "final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv",
    "perpetration": VERSION_FINAL_DIR / "final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv",
    "overlap": VERSION_FINAL_DIR / (
        "final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

PRED_PATH = PREDICTION_FILES[OUTCOME]

print("OUTCOME:", OUTCOME)
print("VERSION_FINAL_DIR:", VERSION_FINAL_DIR)
print("PRED_PATH:", PRED_PATH)
print("PRED exists:", PRED_PATH.exists())
print("EXPORT_DIR:", EXPORT_DIR)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def get_first_col(df, colname):
    x = df[colname]
    if isinstance(x, pd.DataFrame):
        print(f"WARNING duplicated column '{colname}'. Using first occurrence.")
        x = x.iloc[:, 0]
    return x


def load_predictions(path, outcome):
    raw = pd.read_csv(path)

    if raw.columns.duplicated().any():
        print("WARNING duplicated columns in prediction file:")
        print(raw.columns[raw.columns.duplicated()].tolist())
        raw = raw.loc[:, ~raw.columns.duplicated()].copy()

    idx_col = detect_col(raw, ["idx_original", "original_idx", "idx", "index"])

    y_true_col = detect_col(raw, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(raw, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(raw, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    if idx_col is None or y_true_col is None or y_pred_col is None:
        raise ValueError(f"Cannot detect prediction columns. Found: {list(raw.columns)}")

    out = pd.DataFrame({
        "idx_original": pd.to_numeric(get_first_col(raw, idx_col), errors="coerce"),
        "y_true_official": pd.to_numeric(get_first_col(raw, y_true_col), errors="coerce"),
        "y_pred": pd.to_numeric(get_first_col(raw, y_pred_col), errors="coerce"),
    })

    if y_prob_col is not None:
        out["y_prob"] = pd.to_numeric(get_first_col(raw, y_prob_col), errors="coerce")

    out = out.dropna(subset=["idx_original", "y_true_official", "y_pred"]).copy()
    out["idx_original"] = out["idx_original"].astype(int)
    out["y_true_official"] = (out["y_true_official"] > 0).astype(int)
    out["y_pred"] = (out["y_pred"] > 0).astype(int)

    return out


def make_y_from_counts(df, outcome, threshold=1):
    if outcome == "victimization":
        return (pd.to_numeric(df[V_COL], errors="coerce") >= threshold).astype(int)

    if outcome == "perpetration":
        return (pd.to_numeric(df[P_COL], errors="coerce") >= threshold).astype(int)

    if outcome == "overlap":
        return (
            (pd.to_numeric(df[V_COL], errors="coerce") >= threshold)
            &
            (pd.to_numeric(df[P_COL], errors="coerce") >= threshold)
        ).astype(int)

    raise ValueError(outcome)


def align_candidate_to_predictions(candidate_df, pred_idx):
    """
    Try several alignment modes:
    1. loc by index
    2. iloc by integer positions
    3. merge by idx_original column if present
    """
    trials = []

    dfc = candidate_df.copy()

    # Remove duplicated columns
    if dfc.columns.duplicated().any():
        dfc = dfc.loc[:, ~dfc.columns.duplicated()].copy()

    pred_idx = pd.Series(pred_idx).astype(int)

    # Trial 1: .loc
    try:
        if set(pred_idx).issubset(set(dfc.index.astype(int))):
            aligned = dfc.loc[pred_idx].reset_index(drop=True).copy()
            trials.append(("loc_index", aligned))
    except Exception:
        pass

    # Trial 2: .iloc
    try:
        if pred_idx.min() >= 0 and pred_idx.max() < len(dfc):
            aligned = dfc.iloc[pred_idx.to_numpy()].reset_index(drop=True).copy()
            trials.append(("iloc_position", aligned))
    except Exception:
        pass

    # Trial 3: idx_original column
    try:
        if "idx_original" in dfc.columns:
            tmp = pd.DataFrame({"idx_original": pred_idx})
            aligned = tmp.merge(dfc, on="idx_original", how="left")
            aligned = aligned.reset_index(drop=True).copy()
            trials.append(("merge_idx_original", aligned))
    except Exception:
        pass

    return trials


# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

pred = load_predictions(PRED_PATH, OUTCOME)

print("\nPrediction file loaded")
print(pred.head())
print("n:", len(pred))
print("idx min/max:", pred["idx_original"].min(), pred["idx_original"].max())
print("official positives:", int(pred["y_true_official"].sum()))
print("predicted positives:", int(pred["y_pred"].sum()))


# ------------------------------------------------------------
# Find candidate DataFrames in notebook memory
# ------------------------------------------------------------

candidate_dfs = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns.astype(str))
        has_counts = V_COL in cols and P_COL in cols
        has_subgroups = any(c in cols for c in SUBGROUP_COLS)

        if has_counts or has_subgroups:
            candidate_dfs.append((name, obj))

print("\nCandidate DataFrames found:")
for name, obj in candidate_dfs:
    print(name, obj.shape, list(obj.columns[:12]))


# ------------------------------------------------------------
# Score count candidates
# ------------------------------------------------------------

score_rows = []
best = None

for name, dfc in candidate_dfs:
    if V_COL not in dfc.columns or P_COL not in dfc.columns:
        continue

    trials = align_candidate_to_predictions(dfc, pred["idx_original"])

    for mode, aligned in trials:
        if aligned[[V_COL, P_COL]].isna().any().any():
            match_rate = np.nan
            positives_calc = np.nan
        else:
            y_calc = make_y_from_counts(aligned, OUTCOME, threshold=1)
            match_rate = float((y_calc.to_numpy() == pred["y_true_official"].to_numpy()).mean())
            positives_calc = int(y_calc.sum())

        row = {
            "df_name": name,
            "shape": str(dfc.shape),
            "mode": mode,
            "match_rate_count_ge1_vs_official_ytrue": match_rate,
            "official_positives": int(pred["y_true_official"].sum()),
            "count_ge1_positives": positives_calc,
        }

        score_rows.append(row)

        if best is None or (pd.notna(match_rate) and match_rate > best["match_rate"]):
            best = {
                "df_name": name,
                "df": dfc,
                "mode": mode,
                "aligned": aligned,
                "match_rate": match_rate,
            }

score_df = pd.DataFrame(score_rows).sort_values(
    "match_rate_count_ge1_vs_official_ytrue",
    ascending=False
)

print("\n=== COUNT ALIGNMENT SCORES ===")
print(score_df.to_string(index=False))

if best is None:
    raise RuntimeError("No DataFrame with V.SUM.TOTAL and P.SUM.TOTAL found in memory.")

print("\nBEST COUNT ALIGNMENT:")
print("df:", best["df_name"])
print("mode:", best["mode"])
print("match_rate:", best["match_rate"])

if best["match_rate"] < 0.999:
    raise RuntimeError(
        "No aligned in-memory DataFrame reproduced official y_true using V/P counts. "
        "This notebook probably no longer has the filtered analytic target DataFrame in memory. "
        "Add V.SUM.TOTAL and P.SUM.TOTAL to the working DataFrame before train_test_split and rerun."
    )

counts_aligned = best["aligned"].reset_index(drop=True).copy()


# ------------------------------------------------------------
# Find subgroup candidate using same alignment logic
# ------------------------------------------------------------

subgroup_aligned = None
subgroup_source = None
subgroup_mode = None

# Prefer same DataFrame if it already contains subgroup columns
if any(c in counts_aligned.columns for c in SUBGROUP_COLS):
    subgroup_aligned = counts_aligned.copy()
    subgroup_source = best["df_name"]
    subgroup_mode = best["mode"]

else:
    for name, dfc in candidate_dfs:
        if not any(c in dfc.columns for c in SUBGROUP_COLS):
            continue

        trials = align_candidate_to_predictions(dfc, pred["idx_original"])

        for mode, aligned in trials:
            available = [c for c in SUBGROUP_COLS if c in aligned.columns]

            if len(available) >= 3 and len(aligned) == len(pred):
                subgroup_aligned = aligned.reset_index(drop=True).copy()
                subgroup_source = name
                subgroup_mode = mode
                break

        if subgroup_aligned is not None:
            break

print("\nSUBGROUP ALIGNMENT:")
print("source:", subgroup_source)
print("mode:", subgroup_mode)

if subgroup_aligned is None:
    print("WARNING: no subgroup DataFrame aligned. Export will contain counts only.")
    subgroup_aligned = pd.DataFrame(index=np.arange(len(pred)))


# ------------------------------------------------------------
# Build final aligned export
# ------------------------------------------------------------

export_df = pred.reset_index(drop=True).copy()

export_df[V_COL] = pd.to_numeric(counts_aligned[V_COL], errors="coerce").to_numpy()
export_df[P_COL] = pd.to_numeric(counts_aligned[P_COL], errors="coerce").to_numpy()

for col in SUBGROUP_COLS:
    if col in subgroup_aligned.columns:
        export_df[col] = subgroup_aligned[col].to_numpy()

# Add useful derived targets
export_df["victim_count_ge1"] = (export_df[V_COL] >= 1).astype(int)
export_df["victim_count_ge2"] = (export_df[V_COL] >= 2).astype(int)
export_df["victim_count_ge3"] = (export_df[V_COL] >= 3).astype(int)

export_df["perp_count_ge1"] = (export_df[P_COL] >= 1).astype(int)
export_df["perp_count_ge2"] = (export_df[P_COL] >= 2).astype(int)
export_df["perp_count_ge3"] = (export_df[P_COL] >= 3).astype(int)

export_df["overlap_count_ge1"] = ((export_df[V_COL] >= 1) & (export_df[P_COL] >= 1)).astype(int)
export_df["overlap_count_ge2"] = ((export_df[V_COL] >= 2) & (export_df[P_COL] >= 2)).astype(int)
export_df["overlap_count_ge3"] = ((export_df[V_COL] >= 3) & (export_df[P_COL] >= 3)).astype(int)

# Final validation
if OUTCOME == "victimization":
    validation_y = export_df["victim_count_ge1"]
elif OUTCOME == "perpetration":
    validation_y = export_df["perp_count_ge1"]
else:
    validation_y = export_df["overlap_count_ge1"]

validation_match = (validation_y.to_numpy() == export_df["y_true_official"].to_numpy()).mean()

print("\nFINAL VALIDATION")
print("validation_match_count_ge1_vs_y_true_official:", validation_match)
print("official positives:", int(export_df["y_true_official"].sum()))
print("count_ge1 positives:", int(validation_y.sum()))

if validation_match < 0.999:
    raise RuntimeError("Final validation failed. Do not use this export.")

out_path = EXPORT_DIR / f"{OUTCOME}_aligned_test_rowlevel.csv"
export_df.to_csv(out_path, index=False)

print("\nSaved aligned row-level export to:")
print(out_path)
print("\nColumns exported:")
print(list(export_df.columns))

OUTCOME: perpetration
VERSION_FINAL_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final
PRED_PATH: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
PRED exists: True
EXPORT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/aligned_test_exports

Prediction file loaded
   idx_original  y_true_official  y_pred    y_prob
0             1                1       1  0.554482
1             2                0       1  0.636366
2             5                0       1  0.659102
3            11                0       1  0.502891
4            13                0       1  0.606305
n: 942
idx min/max: 1 3759
official positives: 221
predicted positives: 706

Candidate DataFrames found:
feat_df (4024, 29) ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CO

In [6]:
# ============================================================
# EXPORT PCA ARTIFACTS FOR SUPPLEMENTARY VARIANCE/BACK-PROJECTION
# Run inside each FINAL notebook after PCA has been fitted
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# CONFIG — change by notebook
# ------------------------------------------------------------

# OUTCOME = "victimization"
OUTCOME = "perpetration"
# OUTCOME = "overlap"

# ------------------------------------------------------------
# Locate Version_Final
# ------------------------------------------------------------

VERSION_FINAL_DIR = Path.cwd().resolve()
while VERSION_FINAL_DIR.name != "Version_Final" and VERSION_FINAL_DIR.parent != VERSION_FINAL_DIR:
    VERSION_FINAL_DIR = VERSION_FINAL_DIR.parent

if VERSION_FINAL_DIR.name != "Version_Final":
    raise RuntimeError("Could not locate Version_Final directory.")

EXPORT_DIR = VERSION_FINAL_DIR / "final_pca_artifacts" / OUTCOME
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTCOME:", OUTCOME)
print("VERSION_FINAL_DIR:", VERSION_FINAL_DIR)
print("EXPORT_DIR:", EXPORT_DIR)


# ------------------------------------------------------------
# Find fitted PCA object in memory
# ------------------------------------------------------------

pca_candidates = []

for name, obj in list(globals().items()):
    if hasattr(obj, "components_") and hasattr(obj, "explained_variance_ratio_"):
        try:
            shape = obj.components_.shape
            evr_len = len(obj.explained_variance_ratio_)
            pca_candidates.append((name, obj, shape, evr_len))
        except Exception:
            pass

print("\nPCA candidates found:")
for name, obj, shape, evr_len in pca_candidates:
    print(name, type(obj), shape, evr_len)

if len(pca_candidates) == 0:
    raise RuntimeError(
        "No fitted PCA object found in memory. "
        "Run this cell immediately after the PCA fit/transform cell."
    )

# Prefer common names
preferred_names = ["pca", "modelo_pca", "pca_model", "model_pca", "pca_final"]
selected = None

for pref in preferred_names:
    for name, obj, shape, evr_len in pca_candidates:
        if name.lower() == pref.lower():
            selected = (name, obj)
            break
    if selected is not None:
        break

if selected is None:
    selected = (pca_candidates[0][0], pca_candidates[0][1])

pca_name, pca_obj = selected

print("\nSelected PCA object:", pca_name)
print("components shape:", pca_obj.components_.shape)
print("explained variance retained:", float(np.sum(pca_obj.explained_variance_ratio_)))


# ------------------------------------------------------------
# Find feature names used before PCA
# ------------------------------------------------------------

feature_names = None

# 1. Best case: PCA has sklearn feature_names_in_
if hasattr(pca_obj, "feature_names_in_"):
    feature_names = list(pca_obj.feature_names_in_)
    print("Feature names taken from pca.feature_names_in_")

# 2. Otherwise search DataFrames with same number of columns as PCA input
if feature_names is None:
    n_features = pca_obj.components_.shape[1]

    df_candidates = []

    for name, obj in list(globals().items()):
        if isinstance(obj, pd.DataFrame) and obj.shape[1] == n_features:
            df_candidates.append((name, obj.shape, list(obj.columns)))

    print("\nDataFrame candidates with matching n_features:")
    for name, shape, cols in df_candidates:
        print(name, shape, cols[:8])

    preferred_df_names = [
        "X_train_scaled",
        "X_scaled_train",
        "df_scaled_train",
        "X_train_pre_pca",
        "df_train_scaled",
        "X_train",
        "df_model",
        "df_features",
        "X"
    ]

    selected_df = None

    for pref in preferred_df_names:
        for name, shape, cols in df_candidates:
            if name.lower() == pref.lower():
                selected_df = name
                break
        if selected_df is not None:
            break

    if selected_df is None and len(df_candidates) > 0:
        selected_df = df_candidates[0][0]

    if selected_df is not None:
        feature_names = list(globals()[selected_df].columns)
        print("Feature names taken from DataFrame:", selected_df)

# 3. Last fallback: lista_global_vars.csv if same number of columns
if feature_names is None:
    features_path = VERSION_FINAL_DIR / "data/lista_global_vars.csv"
    features_df = pd.read_csv(features_path)
    n_features = pca_obj.components_.shape[1]

    if features_df.shape[1] == n_features:
        feature_names = list(features_df.columns)
        print("Feature names taken from lista_global_vars.csv")
    else:
        raise RuntimeError(
            f"Could not infer feature names. PCA n_features={n_features}, "
            f"lista_global_vars columns={features_df.shape[1]}. "
            "Run the cell immediately after creating the pre-PCA train matrix."
        )

if len(feature_names) != pca_obj.components_.shape[1]:
    raise RuntimeError(
        f"Feature names length {len(feature_names)} does not match PCA input features "
        f"{pca_obj.components_.shape[1]}"
    )

print("\nNumber of feature names:", len(feature_names))
print("First feature names:", feature_names[:10])


# ------------------------------------------------------------
# Save PCA object and feature names
# ------------------------------------------------------------

joblib.dump(pca_obj, EXPORT_DIR / "modelo_pca.joblib")

pd.DataFrame({"feature": feature_names}).to_csv(
    EXPORT_DIR / "pca_input_feature_names.csv",
    index=False
)

# ------------------------------------------------------------
# PCA variance table
# ------------------------------------------------------------

explained = np.asarray(pca_obj.explained_variance_ratio_)
cumulative = np.cumsum(explained)

variance_df = pd.DataFrame({
    "outcome": OUTCOME,
    "component": [f"PC{i+1}" for i in range(len(explained))],
    "component_number": np.arange(1, len(explained) + 1),
    "explained_variance_ratio": explained,
    "explained_variance_percent": explained * 100,
    "cumulative_variance_ratio": cumulative,
    "cumulative_variance_percent": cumulative * 100,
})

variance_df.to_csv(EXPORT_DIR / "pca_variance.csv", index=False)

# ------------------------------------------------------------
# PCA loadings table
# ------------------------------------------------------------

if hasattr(pca_obj, "explained_variance_"):
    loadings = pca_obj.components_.T * np.sqrt(np.asarray(pca_obj.explained_variance_))
else:
    loadings = pca_obj.components_.T

loading_rows = []

for pc_idx in range(pca_obj.components_.shape[0]):
    for feat_idx, feat in enumerate(feature_names):
        loading = float(loadings[feat_idx, pc_idx])
        loading_rows.append({
            "outcome": OUTCOME,
            "component": f"PC{pc_idx + 1}",
            "component_number": pc_idx + 1,
            "feature": feat,
            "loading": loading,
            "abs_loading": abs(loading),
            "explained_variance_ratio_component": explained[pc_idx],
            "explained_variance_percent_component": explained[pc_idx] * 100,
            "cumulative_variance_percent_at_component": cumulative[pc_idx] * 100,
        })

loadings_df = pd.DataFrame(loading_rows)

loadings_df.to_csv(EXPORT_DIR / "pca_loadings.csv", index=False)

top_loadings_df = (
    loadings_df
    .sort_values(["component_number", "abs_loading"], ascending=[True, False])
    .groupby("component_number")
    .head(8)
    .reset_index(drop=True)
)

top_loadings_df.to_csv(EXPORT_DIR / "pca_top_loadings_by_component.csv", index=False)

print("\nSaved PCA artifacts:")
print(EXPORT_DIR / "modelo_pca.joblib")
print(EXPORT_DIR / "pca_input_feature_names.csv")
print(EXPORT_DIR / "pca_variance.csv")
print(EXPORT_DIR / "pca_loadings.csv")
print(EXPORT_DIR / "pca_top_loadings_by_component.csv")

print("\n=== PCA VARIANCE SUMMARY ===")
print(variance_df.tail(1).to_string(index=False))

print("\n=== TOP LOADINGS PREVIEW ===")
print(top_loadings_df.head(20).to_string(index=False))

OUTCOME: perpetration
VERSION_FINAL_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final
EXPORT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_pca_artifacts/perpetration

PCA candidates found:
pca <class 'sklearn.decomposition._pca.PCA'> (27, 27) 27

Selected PCA object: pca
components shape: (27, 27)
explained variance retained: 0.9999999999999999

DataFrame candidates with matching n_features:
df_features (3767, 27) ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2']
X_train_raw (2825, 27) ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2']
X_val_raw (942, 27) ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2']
dfc (942, 27) ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2']
Feature names taken from DataFrame: df_feat